In [9]:
# 공통 라이브러리 import

from pathlib import Path
import numpy as np
import pandas as pd

import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')# 데이터 경로 설정 및 파일 존재 여부 확인

DATA_DIR = Path('./type3_sample_csvs')  # 필요시 수정

required_files = [
    'churn_logit1.csv',
    'loan_default_logit2.csv'
]

print("=== 파일 확인 ===")
for f in required_files:
    file_path = DATA_DIR / f
    print(f"{f}: {'존재함' if file_path.exists() else '없음'}")

=== 파일 확인 ===
churn_logit1.csv: 존재함
loan_default_logit2.csv: 존재함


## 2. 공통 해석 함수 만들기

아래 함수는 로지스틱 회귀 결과를 조금 더 보기 쉽게 정리해 준다.

### 출력 내용
- 회귀계수
- p-value
- 오즈비
- 신뢰구간
- 예측확률
- accuracy / error rate
- confusion matrix
- log-likelihood
- residual deviance

In [8]:
def logistic_analysis_report(model, df, y_col, threshold=0.5):
    result_table = pd.DataFrame({
        'coef': model.params,
        'p_value': model.pvalues,
        'odds_ratio': np.exp(model.params)
    })

    conf = model.conf_int()
    conf.columns = ['2.5%', '97.5%']
    conf['OR_2.5%'] = np.exp(conf['2.5%'])
    conf['OR_97.5%'] = np.exp(conf['97.5%'])

    df_result = df.copy()
    df_result['pred_prob'] = model.predict(df_result)
    df_result['pred_class'] = (df_result['pred_prob'] >= threshold).astype(int)

    acc = accuracy_score(df_result[y_col], df_result['pred_class'])
    err = 1 - acc
    cm = confusion_matrix(df_result[y_col], df_result['pred_class'])

    llf = model.llf
    llnull = model.llnull
    pseudo_r2 = model.prsquared
    residual_deviance = -2 * llf

    print("=== 회귀 요약표 ===")
    print(model.summary())

    print("=== 계수 / p-value / 오즈비 ===")
    print(result_table)

    print("=== 95% 신뢰구간 및 오즈비 신뢰구간 ===")
    print(conf)

    print("=== 모형 적합도 ===")
    print(f"log-likelihood      : {llf:.4f}")
    print(f"null log-likelihood : {llnull:.4f}")
    print(f"pseudo R^2          : {pseudo_r2:.4f}")
    print(f"residual deviance   : {residual_deviance:.4f}")

    print("=== 분류 성능 ===")
    print(f"accuracy   : {acc:.4f}")
    print(f"error rate : {err:.4f}")

    print("=== confusion matrix ===")
    print(cm)

    print("=== 예측확률 상위 10개 ===")
    print(df_result[[y_col, 'pred_prob', 'pred_class']].head(10))

    return result_table, conf, df_result, acc, err, residual_deviance

# 실습 1. churn_logit1.csv 분석

## 문제 상황
구독형 서비스 기업이 고객 이탈 여부(`churn`)를 예측하고자 한다.  
설명변수는 다음과 같다.

- `age`: 고객 나이
- `usage_hour`: 월 이용시간
- `complaint_cnt`: 최근 불만 건수

## 분석 목표
- 어떤 변수가 고객 이탈에 영향을 주는지 확인한다.
- 각 변수의 방향성과 유의성을 해석한다.
- 오즈비를 이용해 실무적으로 설명한다.

## 가설적 해석 방향
- `complaint_cnt`가 증가하면 이탈 가능성이 커질 수 있다.
- `usage_hour`가 많을수록 서비스 충성도가 높아 이탈 가능성이 낮아질 수 있다.

In [10]:
# churn_logit1.csv 불러오기 및 데이터 확인

churn_df = pd.read_csv(DATA_DIR / 'churn_logit1.csv')

print("=== 데이터 상위 5행 ===")
display(churn_df.head())

print("=== 데이터 구조 ===")
print(churn_df.info())

print("=== 기술통계 ===")
display(churn_df.describe())

print("=== 종속변수 분포 ===")
display(churn_df['churn'].value_counts().sort_index())

# 로지스틱 회귀 적합: churn ~ age + usage_hour + complaint_cnt

churn_model = smf.logit(
    formula='churn ~ age + usage_hour + complaint_cnt',
    data=churn_df
).fit()

churn_result_table, churn_conf, churn_pred_df, churn_acc, churn_err, churn_dev = logistic_analysis_report(
    churn_model, churn_df, 'churn'
)

# 변수별 해석 문장 자동 출력

print("=== 변수별 해석 ===")
for var in ['age', 'usage_hour', 'complaint_cnt']:
    coef = churn_model.params[var]
    pval = churn_model.pvalues[var]
    odds = np.exp(coef)

    direction = "증가" if coef > 0 else "감소"

    print(f"[{var}]")
    print(f"- 계수: {coef:.4f}")
    print(f"- p-value: {pval:.6f}")
    print(f"- 오즈비: {odds:.4f}")

    if pval < 0.05:
        print(f"→ {var}는 유의한 변수이다.")
    else:
        print(f"→ {var}는 유의하다고 보기 어렵다.")

    print(f"→ {var}가 1단위 증가할 때 이탈 odds는 약 {odds:.4f}배가 되며, 방향은 {direction}이다.")

=== 데이터 상위 5행 ===


,churn,age,usage_hour,complaint_cnt
0,0,29,12.0700,0
1,0,20,70.0500,2
2,0,23,58.0900,0
3,0,37,67.2500,2
4,0,54,78.1900,0


=== 데이터 구조 ===
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   churn          500 non-null    int64  
 1   age            500 non-null    int64  
 2   usage_hour     500 non-null    float64
 3   complaint_cnt  500 non-null    int64  
dtypes: float64(1), int64(3)
memory usage: 15.8 KB
None
=== 기술통계 ===


,churn,age,usage_hour,complaint_cnt
count,500.0000,500.0000,500.0000,500.0000
mean,0.0840,39.7640,43.5970,1.4880
std,0.2777,12.1120,21.4417,1.2218
min,0.0000,20.0000,5.0400,0.0000
25%,0.0000,28.7500,24.0700,1.0000
50%,0.0000,39.0000,42.6200,1.0000
75%,0.0000,51.0000,62.7725,2.0000
max,1.0000,60.0000,79.9100,6.0000


=== 종속변수 분포 ===


churn
0    458
1     42
Name: count, dtype: int64

Optimization terminated successfully.
         Current function value: 0.237853
         Iterations 7
=== 회귀 요약표 ===
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  500
Model:                          Logit   Df Residuals:                      496
Method:                           MLE   Df Model:                            3
Date:                Wed, 27 May 2026   Pseudo R-squ.:                  0.1754
Time:                        10:43:58   Log-Likelihood:                -118.93
converged:                       True   LL-Null:                       -144.22
Covariance Type:            nonrobust   LLR p-value:                 6.016e-11
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -1.2848      0.694     -1.852      0.064      -2.645       0.075
age  

# 실습 2. loan_default_logit2.csv 분석

## 문제 상황
은행이 고객의 대출 연체 여부를 예측하고자 한다.  
설명변수는 다음과 같다.

- `income`: 소득
- `debt_ratio`: 부채비율
- `late_cnt`: 과거 연체 횟수

종속변수는 `default`이며, 1이면 연체 발생, 0이면 정상 상환으로 본다.

## 분석 목표
- 어떤 변수가 연체 여부에 영향을 주는지 확인한다.
- 각 변수의 오즈비를 통해 실무적으로 해석한다.
- log-likelihood, residual deviance, accuracy, error rate를 확인한다.

In [11]:
# loan_default_logit2.csv 불러오기 및 데이터 확인

loan_df = pd.read_csv(DATA_DIR / 'loan_default_logit2.csv')

print("=== 원본 데이터 상위 5행 ===")
display(loan_df.head())

print("=== 데이터 구조 ===")
print(loan_df.info())

print("=== 기술통계 ===")
display(loan_df.describe())

print("=== 종속변수 분포 ===")
display(loan_df['default'].value_counts().sort_index())

# 'default'는 Python 예약어와 헷갈릴 수 있으므로 안전하게 이름 변경
loan_df = loan_df.rename(columns={'default': 'default_flag'})

print("변경된 컬럼명:")
print(loan_df.columns.tolist())

# 로지스틱 회귀 적합: default_flag ~ income + debt_ratio + late_cnt

loan_model = smf.logit(
    formula='default_flag ~ income + debt_ratio + late_cnt',
    data=loan_df
).fit()

loan_result_table, loan_conf, loan_pred_df, loan_acc, loan_err, loan_dev = logistic_analysis_report(
    loan_model, loan_df, 'default_flag'
)

# 변수별 해석 문장 자동 출력

print("=== 변수별 해석 ===")
for var in ['income', 'debt_ratio', 'late_cnt']:
    coef = loan_model.params[var]
    pval = loan_model.pvalues[var]
    odds = np.exp(coef)

    direction = "증가" if coef > 0 else "감소"

    print(f"[{var}]")
    print(f"- 계수: {coef:.6f}")
    print(f"- p-value: {pval:.6f}")
    print(f"- 오즈비: {odds:.6f}")

    if pval < 0.05:
        print(f"→ {var}는 유의한 변수이다.")
    else:
        print(f"→ {var}는 유의하다고 보기 어렵다.")

    print(f"→ {var}가 1단위 증가할 때 연체 odds는 약 {odds:.6f}배가 되며, 방향은 {direction}이다.")

=== 원본 데이터 상위 5행 ===


,default,income,debt_ratio,late_cnt
0,0,3313.3900,0.2650,2
1,0,4026.6000,0.3050,0
2,0,3262.6800,0.6840,2
3,0,4087.1200,1.0310,0
4,0,2397.1400,0.1780,0


=== 데이터 구조 ===
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   default     600 non-null    int64  
 1   income      600 non-null    float64
 2   debt_ratio  600 non-null    float64
 3   late_cnt    600 non-null    int64  
dtypes: float64(2), int64(2)
memory usage: 18.9 KB
None
=== 기술통계 ===


,default,income,debt_ratio,late_cnt
count,600.0000,600.0000,600.0000,600.0000
mean,0.2000,5320.7539,0.6295,1.1233
std,0.4003,2033.7036,0.3162,1.0504
min,0.0000,1813.8900,0.1010,0.0000
25%,0.0000,3591.5900,0.3600,0.0000
50%,0.0000,5258.1250,0.6190,1.0000
75%,0.0000,6975.5175,0.9040,2.0000
max,1.0000,8993.0900,1.1980,5.0000


=== 종속변수 분포 ===


default
0    480
1    120
Name: count, dtype: int64

변경된 컬럼명:
['default_flag', 'income', 'debt_ratio', 'late_cnt']
Optimization terminated successfully.
         Current function value: 0.426514
         Iterations 6
=== 회귀 요약표 ===
                           Logit Regression Results                           
Dep. Variable:           default_flag   No. Observations:                  600
Model:                          Logit   Df Residuals:                      596
Method:                           MLE   Df Model:                            3
Date:                Wed, 27 May 2026   Pseudo R-squ.:                  0.1477
Time:                        10:44:18   Log-Likelihood:                -255.91
converged:                       True   LL-Null:                       -300.24
Covariance Type:            nonrobust   LLR p-value:                 4.238e-19
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.7772      0.41

In [12]:
# 두 실습 결과 핵심 지표 비교

summary_df = pd.DataFrame({
    'dataset': ['churn_logit1.csv', 'loan_default_logit2.csv'],
    'accuracy': [churn_acc, loan_acc],
    'error_rate': [churn_err, loan_err],
    'log_likelihood': [churn_model.llf, loan_model.llf],
    'pseudo_R2': [churn_model.prsquared, loan_model.prsquared],
    'residual_deviance': [churn_dev, loan_dev]
})

print("=== 두 모형 비교표 ===")
display(summary_df)

=== 두 모형 비교표 ===


,dataset,accuracy,error_rate,log_likelihood,pseudo_R2,residual_deviance
0,churn_logit1.csv,0.9180,0.0820,-118.9267,0.1754,237.8534
1,loan_default_logit2.csv,0.8150,0.1850,-255.9086,0.1477,511.8172
